In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.models.transformer_baseline import (
    IndependentBandTransformerClassifier,
)

from dfm.training.metrics import (
    accuracy_score,
    macro_f1_score,
)

from dfm.training.profiling import count_parameters

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [4]:
outputs_dir = PROJECT_ROOT / "outputs" / "salinas"

split_path = (
    outputs_dir /
    "salinas_spatial_split_seed42.npz"
)

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

Train: 32337
Validation: 10952
Test: 10840


In [5]:
import subprocess
from pathlib import Path

gscvit_dir = (
    PROJECT_ROOT
    / "external"
    / "gscvit"
)

gscvit_dir.mkdir(
    parents=True,
    exist_ok=True,
)

repo_dir = gscvit_dir / "TGRS-GSC-VIT"

if not repo_dir.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/flyzzie/TGRS-GSC-VIT.git",
            str(repo_dir),
        ],
        check=True,
    )

print("Repository:", repo_dir)

Repository: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\external\gscvit\TGRS-GSC-VIT


In [6]:
from pathlib import Path

for p in sorted(repo_dir.rglob("*")):
    if p.is_file():
        print(p.relative_to(repo_dir))

.git\config
.git\description
.git\HEAD
.git\hooks\applypatch-msg.sample
.git\hooks\commit-msg.sample
.git\hooks\fsmonitor-watchman.sample
.git\hooks\post-update.sample
.git\hooks\pre-applypatch.sample
.git\hooks\pre-commit.sample
.git\hooks\pre-merge-commit.sample
.git\hooks\pre-push.sample
.git\hooks\pre-rebase.sample
.git\hooks\pre-receive.sample
.git\hooks\prepare-commit-msg.sample
.git\hooks\push-to-checkout.sample
.git\hooks\sendemail-validate.sample
.git\hooks\update.sample
.git\index
.git\info\exclude
.git\logs\HEAD
.git\logs\refs\heads\master
.git\logs\refs\remotes\origin\HEAD
.git\objects\pack\pack-bb842302ac0ac24d8f469fabf375a0fa859b4ca3.idx
.git\objects\pack\pack-bb842302ac0ac24d8f469fabf375a0fa859b4ca3.pack
.git\objects\pack\pack-bb842302ac0ac24d8f469fabf375a0fa859b4ca3.rev
.git\packed-refs
.git\refs\heads\master
.git\refs\remotes\origin\HEAD
eval.py
main.py
models\__init__.py
models\caevt.py
models\cnn2d.py
models\cnn3d.py
models\gaht.py
models\get_model.py
models\gscvit.p

In [7]:
gscvit_model_file = repo_dir / "models" / "gscvit.py"

print(gscvit_model_file.read_text(encoding="utf-8"))

import random
from functools import partial
import torch
from torch import nn, einsum
from einops import rearrange, repeat
from einops.layers.torch import Rearrange, Reduce
from thop import profile
from timm.models.vision_transformer import _cfg
from torchsummaryX import summary

def cast_tuple(val, length=1):
    return val if isinstance(val, tuple) else ((val,) * length)


# 通道校准策略1
class ChannelAdjustmentLayer1(nn.Module):
    def __init__(self, target_channels=256):
        super(ChannelAdjustmentLayer1, self).__init__()
        self.target_channels = target_channels

    def forward(self, x):
        B, C, H, W = x.size()

        if C == self.target_channels:
            return x

        if C < self.target_channels:
            # 逐个通道复制，放到被复制通道的后面
            num_channels_to_copy = self.target_channels - C
            for i in range(num_channels_to_copy):
                channel_to_copy = torch.randint(0, C, (1,))
                x = torch.cat([x, x[:, channel_to_copy, :, :]], d

In [8]:
get_model_file = repo_dir / "models" / "get_model.py"

print(get_model_file.read_text(encoding="utf-8"))

from .cnn2d import cnn2d
from .sprn import SPRN
from .cnn3d import cnn3d
from .hybridsn import hybridsn
from .spectralformer import spectralformer
from .ssftt import ssftt
from .gaht import gaht
from .gscvit import gscvit
from .morphFormer import morphFormer
from .caevt import caevt
def get_model(model_name, dataset_name, patch_size):
    if model_name == 'cnn2d':
        model = cnn2d(dataset=dataset_name)

    elif model_name == 'sprn':
        model = SPRN(dataset=dataset_name)

    elif model_name == 'cnn3d':
        model = cnn3d(dataset_name, patch_size)

    elif model_name == 'hybridsn':
        model = hybridsn(dataset_name, patch_size)

    elif model_name == 'spectralformer':
        model = spectralformer(dataset_name, patch_size)

    elif model_name == 'ssftt':
        model = ssftt(dataset_name, patch_size)

    elif model_name == 'gaht':
        model = gaht(dataset_name, patch_size)

    elif model_name == 'morphFormer':
        model = morphFormer(16, 80, 10, False, 8

In [9]:
train_file = repo_dir / "train.py"

print(train_file.read_text(encoding="utf-8"))

import os
import numpy as np
from tqdm import tqdm
import torch
from utils.utils import grouper, sliding_window, count_sliding_window


def train(network, optimizer, criterion, train_loader, val_loader, epoch, saving_path, device, scheduler=None):

    best_acc = -0.1
    losses = []

    for e in tqdm(range(1, epoch+1), desc=""):
        network.train()
        for batch_idx, (images, targets) in enumerate(train_loader):
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = network(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if e % 10 == 0 or e == 1:
            mean_losses = np.mean(losses)
            train_info = "train at epoch {}/{}, loss={:.6f}"
            train_info = train_info.format(e, epoch,  mean_losses)
            tqdm.write(train_info)
            losses = []
        else:
            lo

In [17]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


class GSCViTDataset(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x, y = self.base_dataset[idx]

        # Convert NumPy -> PyTorch
        if not torch.is_tensor(x):
            x = torch.from_numpy(x)

        x = x.float()

        # Expected: [204, 15, 15]
        if x.ndim != 3:
            raise ValueError(
                f"Expected [C,H,W], got {x.shape}"
            )

        # Pad 15x15 -> 16x16.
        # This is only an input-interface adaptation;
        # the GSC-ViT architecture remains unchanged.
        if x.shape[-2:] == (15, 15):
            x = F.pad(
                x,
                (0, 1, 0, 1),
                mode="reflect",
            )

        elif x.shape[-2:] != (16, 16):
            raise ValueError(
                f"Unexpected patch size: {x.shape}"
            )

        return x, torch.as_tensor(y).long()

In [23]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))

Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [24]:
train_dataset = SalinasPatchDataset(
    scene=scene,
    indices=train_indices,
    patch_size=15,
    normalize=True,
)

val_dataset = SalinasPatchDataset(
    scene=scene,
    indices=val_indices,
    patch_size=15,
    normalize=True,
)

test_dataset = SalinasPatchDataset(
    scene=scene,
    indices=test_indices,
    patch_size=15,
    normalize=True,
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

32337 10952 10840


In [25]:
gsc_train_dataset = GSCViTDataset(
    train_dataset
)

gsc_val_dataset = GSCViTDataset(
    val_dataset
)

gsc_test_dataset = GSCViTDataset(
    test_dataset
)

print(
    len(gsc_train_dataset),
    len(gsc_val_dataset),
    len(gsc_test_dataset),
)

32337 10952 10840


In [26]:
gsc_train_dataset = GSCViTDataset(train_dataset)
gsc_val_dataset = GSCViTDataset(val_dataset)
gsc_test_dataset = GSCViTDataset(test_dataset)

gsc_batch_size = 64

gsc_train_loader = DataLoader(
    gsc_train_dataset,
    batch_size=gsc_batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

gsc_val_loader = DataLoader(
    gsc_val_dataset,
    batch_size=gsc_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

gsc_test_loader = DataLoader(
    gsc_test_dataset,
    batch_size=gsc_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [27]:
x, y = next(iter(gsc_train_loader))

print("Input :", x.shape)
print("Labels:", y.shape)
print("Bands :", x.shape[1])
print("Spatial:", x.shape[-2:])

Input : torch.Size([64, 204, 16, 16])
Labels: torch.Size([64])
Bands : 204
Spatial: torch.Size([16, 16])


In [29]:
import sys
from pathlib import Path

repo_dir = Path(repo_dir).resolve()

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("GSC-ViT repo:", repo_dir)
print("In sys.path:", str(repo_dir) in sys.path)

GSC-ViT repo: C:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\external\gscvit\TGRS-GSC-VIT
In sys.path: True


In [30]:
import models

print("models imported from:")
print(models.__file__)

models imported from:
C:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\external\gscvit\TGRS-GSC-VIT\models\__init__.py


In [34]:
from models.gscvit import gscvit

gscvit_model = gscvit(
    dataset="sa"
).to(device)

gscvit_model.eval()

print(gscvit_model)

GSCViT(
  (sc): SpectralCalibration(
    (conv): Conv2d(204, 256, kernel_size=(1, 1), stride=(1, 1))
    (bn): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
  )
  (bn_1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu_1): ReLU(inplace=True)
  (layers_trans): ModuleList(
    (0): ModuleList(
      (0): GSC(
        (gpwc): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1), groups=16)
        (gc): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16)
        (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (1): Transformer(
        (layers): ModuleList(
          (0): PreNorm(
            (norm): ChanLayerNorm()
            (fn): GSSA(
              (attend): Sequential(
                (0): Softmax(dim=-1)
                (1): Dropout(p=0.1, inplace=False)
            

In [35]:
x, y = next(iter(gsc_train_loader))

x = x.to(
    device,
    dtype=torch.float32,
)

with torch.no_grad():
    logits = gscvit_model(x)

print("Input :", x.shape)
print("Output:", logits.shape)

Input : torch.Size([64, 204, 16, 16])
Output: torch.Size([64, 16])


In [36]:
def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

gscvit_params = count_trainable_parameters(gscvit_model)

print("GSC-ViT trainable parameters:", gscvit_params)
print("GSC-ViT parameters (M):", gscvit_params / 1e6)

GSC-ViT trainable parameters: 179024
GSC-ViT parameters (M): 0.179024


In [37]:
print("NaN in output:", torch.isnan(logits).any().item())
print("Inf in output:", torch.isinf(logits).any().item())

NaN in output: False
Inf in output: False


In [42]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


def train_gscvit_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    model.train()

    running_loss = 0.0
    total_samples = 0

    for x, y in loader:

        x = x.to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        y = y.to(
            device,
            dtype=torch.long,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        batch_size = y.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

    return running_loss / total_samples


@torch.no_grad()
def evaluate_gscvit(
    model,
    loader,
    device,
):
    model.eval()

    all_predictions = []
    all_targets = []

    for x, y in loader:

        x = x.to(
            device,
            dtype=torch.float32,
            non_blocking=True,
        )

        y = y.to(
            device,
            dtype=torch.long,
            non_blocking=True,
        )

        logits = model(x)

        predictions = logits.argmax(dim=1)

        all_predictions.append(
            predictions.cpu().numpy()
        )

        all_targets.append(
            y.cpu().numpy()
        )

    all_predictions = np.concatenate(
        all_predictions
    )

    all_targets = np.concatenate(
        all_targets
    )

    accuracy = accuracy_score(
        all_targets,
        all_predictions,
    )

    macro_f1 = f1_score(
        all_targets,
        all_predictions,
        average="macro",
        zero_division=0,
    )

    cm = confusion_matrix(
        all_targets,
        all_predictions,
    )

    return (
        accuracy,
        macro_f1,
        cm,
        all_predictions,
    )

In [43]:
from copy import deepcopy

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    gscvit_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
)

EPOCHS = 50

best_gscvit_val_f1 = -1.0
best_gscvit_epoch = 0
best_gscvit_state = None

gscvit_history = []

for epoch in range(1, EPOCHS + 1):

    train_loss = train_gscvit_one_epoch(
        gscvit_model,
        gsc_train_loader,
        criterion,
        optimizer,
        device,
    )

    val_acc, val_f1, _, _ = evaluate_gscvit(
        gscvit_model,
        gsc_val_loader,
        device,
    )

    scheduler.step()

    gscvit_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    if val_f1 > best_gscvit_val_f1:

        best_gscvit_val_f1 = val_f1
        best_gscvit_epoch = epoch

        best_gscvit_state = deepcopy(
            gscvit_model.state_dict()
        )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
        + ("  <-- BEST" if val_f1 == best_gscvit_val_f1 else "")
    )

print()
print("Best epoch:", best_gscvit_epoch)
print(
    "Best validation Macro-F1:",
    best_gscvit_val_f1,
)

Epoch 01/50 | Loss: 0.0800 | Val Acc: 0.7980 | Val Macro-F1: 0.8362  <-- BEST
Epoch 02/50 | Loss: 0.0333 | Val Acc: 0.9080 | Val Macro-F1: 0.9240  <-- BEST
Epoch 03/50 | Loss: 0.0302 | Val Acc: 0.8960 | Val Macro-F1: 0.9457  <-- BEST
Epoch 04/50 | Loss: 0.0190 | Val Acc: 0.8210 | Val Macro-F1: 0.8471
Epoch 05/50 | Loss: 0.0167 | Val Acc: 0.9059 | Val Macro-F1: 0.9527  <-- BEST
Epoch 06/50 | Loss: 0.0135 | Val Acc: 0.9117 | Val Macro-F1: 0.9180
Epoch 07/50 | Loss: 0.0128 | Val Acc: 0.9339 | Val Macro-F1: 0.9617  <-- BEST
Epoch 08/50 | Loss: 0.0143 | Val Acc: 0.9309 | Val Macro-F1: 0.9634  <-- BEST
Epoch 09/50 | Loss: 0.0164 | Val Acc: 0.9502 | Val Macro-F1: 0.9263
Epoch 10/50 | Loss: 0.0108 | Val Acc: 0.8941 | Val Macro-F1: 0.9485
Epoch 11/50 | Loss: 0.0061 | Val Acc: 0.9000 | Val Macro-F1: 0.9443
Epoch 12/50 | Loss: 0.0094 | Val Acc: 0.8903 | Val Macro-F1: 0.9059
Epoch 13/50 | Loss: 0.0113 | Val Acc: 0.9342 | Val Macro-F1: 0.9472
Epoch 14/50 | Loss: 0.0096 | Val Acc: 0.9159 | Val Macro

In [44]:
from copy import deepcopy

# Restore best validation checkpoint
gscvit_model.load_state_dict(best_gscvit_state)

gscvit_model = gscvit_model.to(device)
gscvit_model.eval()

print("Loaded best GSC-ViT model.")
print("Best epoch:", best_gscvit_epoch)
print("Best validation Macro-F1:", best_gscvit_val_f1)

Loaded best GSC-ViT model.
Best epoch: 34
Best validation Macro-F1: 0.9685648686780536


In [45]:
test_acc, test_f1, test_cm, test_predictions = evaluate_gscvit(
    gscvit_model,
    gsc_test_loader,
    device,
)

print("=" * 60)
print("GSC-ViT SALINAS TEST RESULTS")
print("=" * 60)

print(f"Accuracy : {test_acc:.4f}")
print(f"Macro-F1 : {test_f1:.4f}")

GSC-ViT SALINAS TEST RESULTS
Accuracy : 0.9806
Macro-F1 : 0.9667


In [46]:
from sklearn.metrics import classification_report

test_targets = []

with torch.no_grad():
    for x, y in gsc_test_loader:
        test_targets.append(
            y.numpy()
        )

test_targets = np.concatenate(test_targets)

print(
    classification_report(
        test_targets,
        test_predictions,
        digits=4,
        zero_division=0,
    )
)

              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       909
           1     1.0000    1.0000    1.0000       498
           2     0.9136    1.0000    0.9548       296
           3     1.0000    1.0000    1.0000       295
           4     1.0000    1.0000    1.0000       863
           5     1.0000    1.0000    1.0000       632
           6     1.0000    1.0000    1.0000       120
           7     0.9866    0.9679    0.9772      2899
           8     1.0000    1.0000    1.0000      1177
           9     0.9084    0.8566    0.8817       544
          10     0.5930    1.0000    0.7445        51
          11     1.0000    1.0000    1.0000       448
          12     1.0000    1.0000    1.0000       233
          13     1.0000    0.9474    0.9730       228
          14     0.9127    0.9600    0.9357       675
          15     1.0000    1.0000    1.0000       972

    accuracy                         0.9806     10840
   macro avg     0.9571   

In [47]:
gscvit_best_path = (
    PROJECT_ROOT
    / "results"
    / "gscvit"
    / "gscvit_salinas_best.pth"
)

gscvit_best_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

torch.save(
    best_gscvit_state,
    gscvit_best_path,
)

print("Saved:", gscvit_best_path)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring\results\gscvit\gscvit_salinas_best.pth


In [48]:
import gc
import time
import numpy as np
import torch

# ============================================================
# GSC-ViT BENCHMARK
# ============================================================

# Make sure best model is loaded
gscvit_model.load_state_dict(best_gscvit_state)
gscvit_model = gscvit_model.to(device)
gscvit_model.eval()

# ------------------------------------------------------------
# 1. Parameters
# ------------------------------------------------------------
gscvit_parameters = sum(
    p.numel()
    for p in gscvit_model.parameters()
    if p.requires_grad
)

gscvit_parameters_m = gscvit_parameters / 1e6

print("Trainable parameters:", gscvit_parameters)
print("Parameters (M):", gscvit_parameters_m)

# ------------------------------------------------------------
# 2. Model/state-dict size
# ------------------------------------------------------------
gscvit_state_bytes = sum(
    v.numel() * v.element_size()
    for v in gscvit_model.state_dict().values()
)

gscvit_model_size_mb = (
    gscvit_state_bytes / (1024 ** 2)
)

print(
    f"Raw state-dict size: "
    f"{gscvit_model_size_mb:.3f} MB"
)

# ------------------------------------------------------------
# 3. FLOPs
# ------------------------------------------------------------
from thop import profile

# Use the actual benchmark batch size
BENCH_BATCH_SIZE = 64

x_bench, _ = next(iter(gsc_test_loader))

x_bench = x_bench[:BENCH_BATCH_SIZE].to(
    device,
    dtype=torch.float32,
)

# FLOPs should be measured with gradients disabled
gscvit_model.eval()

with torch.no_grad():
    flops, params_thop = profile(
        gscvit_model,
        inputs=(x_bench,),
        verbose=False,
    )

gscvit_gflops = flops / 1e9

print(f"FLOPs   : {flops:.0f}")
print(f"GFLOPs  : {gscvit_gflops:.6f}")

del x_bench
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Latency / throughput benchmark
# ------------------------------------------------------------

x_bench, _ = next(iter(gsc_test_loader))

x_bench = x_bench[:BENCH_BATCH_SIZE].to(
    device,
    dtype=torch.float32,
)

# Warm-up
with torch.no_grad():
    for _ in range(20):
        _ = gscvit_model(x_bench)

torch.cuda.synchronize()

# ------------------------------------------------------------
# Timed iterations
# ------------------------------------------------------------
NUM_TIMED = 100

times_ms = []

with torch.no_grad():

    for _ in range(NUM_TIMED):

        torch.cuda.synchronize()

        start = time.perf_counter()

        _ = gscvit_model(x_bench)

        torch.cuda.synchronize()

        end = time.perf_counter()

        times_ms.append(
            (end - start) * 1000
        )

times_ms = np.asarray(times_ms)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------
latency_batch_ms = float(
    times_ms.mean()
)

latency_batch_median_ms = float(
    np.median(times_ms)
)

latency_batch_p95_ms = float(
    np.percentile(times_ms, 95)
)

latency_batch_std_ms = float(
    times_ms.std(ddof=1)
)

latency_per_sample_ms = (
    latency_batch_ms / BENCH_BATCH_SIZE
)

throughput_samples_sec = (
    BENCH_BATCH_SIZE
    / (latency_batch_ms / 1000.0)
)

# ------------------------------------------------------------
# Peak GPU memory
# ------------------------------------------------------------
torch.cuda.reset_peak_memory_stats(device)

with torch.no_grad():

    for _ in range(10):
        _ = gscvit_model(x_bench)

torch.cuda.synchronize()

peak_gpu_memory_mb = (
    torch.cuda.max_memory_allocated(device)
    / (1024 ** 2)
)

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
gscvit_benchmark = {
    "model": "GSC-ViT",
    "accuracy": float(test_acc),
    "macro_f1": float(test_f1),
    "parameters": int(gscvit_parameters),
    "parameters_m": float(gscvit_parameters_m),
    "model_size_mb": float(gscvit_model_size_mb),
    "flops": float(flops),
    "gflops": float(gscvit_gflops),
    "peak_gpu_memory_mb": float(
        peak_gpu_memory_mb
    ),
    "batch_size": BENCH_BATCH_SIZE,
    "latency_batch_ms": latency_batch_ms,
    "latency_batch_median_ms": latency_batch_median_ms,
    "latency_batch_p95_ms": latency_batch_p95_ms,
    "latency_batch_std_ms": latency_batch_std_ms,
    "latency_per_sample_ms": latency_per_sample_ms,
    "throughput_samples_sec": throughput_samples_sec,
}

print()
print("=" * 60)
print("GSC-ViT BENCHMARK RESULTS")
print("=" * 60)

for key, value in gscvit_benchmark.items():
    print(f"{key:30s}: {value}")

Trainable parameters: 179024
Parameters (M): 0.179024
Raw state-dict size: 0.690 MB
FLOPs   : 1694451712
GFLOPs  : 1.694452

GSC-ViT BENCHMARK RESULTS
model                         : GSC-ViT
accuracy                      : 0.9806273062730627
macro_f1                      : 0.9666879812641892
parameters                    : 179024
parameters_m                  : 0.179024
model_size_mb                 : 0.6898040771484375
flops                         : 1694451712.0
gflops                        : 1.694451712
peak_gpu_memory_mb            : 74.94091796875
batch_size                    : 64
latency_batch_ms              : 14.937459999346174
latency_batch_median_ms       : 14.731600000231992
latency_batch_p95_ms          : 16.599684998072917
latency_batch_std_ms          : 0.8254062909507822
latency_per_sample_ms         : 0.23339781248978397
throughput_samples_sec        : 4284.530301858638


In [51]:
import pandas as pd
from pathlib import Path

# ============================================================
# SALINAS SOTA BENCHMARK RESULTS
# ============================================================

results = [
    {
        "model": "Hybrid Spatial-Spectral",
        "accuracy": 0.9810,
        "macro_f1": 0.9639,
        "parameters": 605712,
        "parameters_m": 0.6057,
        "model_size_mb": 2.330,
        "gflops": 0.0821,
        "peak_gpu_memory_mb": 252.45,
        "batch_size": 128,
        "latency_batch_ms": 43.815,
        "latency_batch_median_ms": 44.874,
        "latency_batch_p95_ms": 45.744,
        "latency_batch_std_ms": 2.002,
        "latency_per_sample_ms": 0.3423,
        "throughput_samples_sec": 2921.36,
    },

    {
        "model": "SpectralFormer (Official)",
        "accuracy": 0.9274,
        "macro_f1": 0.9187,
        "parameters": 399381,
        "parameters_m": 0.3994,
        "model_size_mb": 1.553,
        "gflops": 0.0433,
        "peak_gpu_memory_mb": 370.26,
        "batch_size": 128,
        "latency_batch_ms": 67.641,
        "latency_batch_median_ms": 65.251,
        "latency_batch_p95_ms": 71.436,
        "latency_batch_std_ms": 18.100,
        "latency_per_sample_ms": 0.5284,
        "throughput_samples_sec": 1892.35,
    },

    {
        "model": "SSFTT (Official)",
        "accuracy": 0.9815,
        "macro_f1": 0.9541,
        "parameters": 153224,
        "parameters_m": 0.1532,
        "model_size_mb": 0.598,
        "gflops": 0.0169,
        "peak_gpu_memory_mb": 47.74,
        "batch_size": 128,
        "latency_batch_ms": 8.623,
        "latency_batch_median_ms": 5.602,
        "latency_batch_p95_ms": 10.491,
        "latency_batch_std_ms": 13.052,
        "latency_per_sample_ms": 0.0674,
        "throughput_samples_sec": 14844.05,
    },

    {
        "model": "MorphFormer",
        "accuracy": 0.9707564576,
        "macro_f1": 0.9619364919,
        "parameters": 280752,
        "parameters_m": 0.280752,
        "model_size_mb": 1.072555542,
        "gflops": 8.522272768,
        "peak_gpu_memory_mb": 518.7836914,
        "batch_size": 128,
        "latency_batch_ms": 137.57408,
        "latency_batch_median_ms": 133.4902,
        "latency_batch_p95_ms": 159.42666,
        "latency_batch_std_ms": 16.447802,
        "latency_per_sample_ms": 1.0747975,
        "throughput_samples_sec": 930.407821,
    },

    {
        "model": "GSC-ViT",
        "accuracy": 0.9806273063,
        "macro_f1": 0.9666879813,
        "parameters": 179024,
        "parameters_m": 0.179024,
        "model_size_mb": 0.6898040771,
        "gflops": 1.694451712,
        "peak_gpu_memory_mb": 74.94091797,
        "batch_size": 64,
        "latency_batch_ms": 14.93746,
        "latency_batch_median_ms": 14.7316,
        "latency_batch_p95_ms": 16.599685,
        "latency_batch_std_ms": 0.8254063,
        "latency_per_sample_ms": 0.2333978125,
        "throughput_samples_sec": 4284.530302,
    },
]


# ============================================================
# CREATE DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

print("=" * 100)
print("SALINAS SOTA BENCHMARK RESULTS")
print("=" * 100)

display(results_df)


# ============================================================
# RANKINGS
# ============================================================

results_df["accuracy_rank"] = (
    results_df["accuracy"]
    .rank(ascending=False, method="min")
    .astype(int)
)

results_df["macro_f1_rank"] = (
    results_df["macro_f1"]
    .rank(ascending=False, method="min")
    .astype(int)
)

results_df["latency_rank"] = (
    results_df["latency_per_sample_ms"]
    .rank(ascending=True, method="min")
    .astype(int)
)

results_df["throughput_rank"] = (
    results_df["throughput_samples_sec"]
    .rank(ascending=False, method="min")
    .astype(int)
)

results_df["parameters_rank"] = (
    results_df["parameters"]
    .rank(ascending=True, method="min")
    .astype(int)
)

results_df["gflops_rank"] = (
    results_df["gflops"]
    .rank(ascending=True, method="min")
    .astype(int)
)


# ============================================================
# MACRO-F1 RANKING
# ============================================================

print()
print("=" * 100)
print("RANKING BY MACRO-F1")
print("=" * 100)

macro_f1_ranking = results_df.sort_values(
    "macro_f1",
    ascending=False
)

display(
    macro_f1_ranking[
        [
            "model",
            "accuracy",
            "macro_f1",
            "parameters_m",
            "gflops",
            "latency_per_sample_ms",
            "throughput_samples_sec",
        ]
    ]
)


# ============================================================
# ACCURACY RANKING
# ============================================================

print()
print("=" * 100)
print("RANKING BY ACCURACY")
print("=" * 100)

accuracy_ranking = results_df.sort_values(
    "accuracy",
    ascending=False
)

display(
    accuracy_ranking[
        [
            "model",
            "accuracy",
            "macro_f1",
            "parameters_m",
            "gflops",
        ]
    ]
)


# ============================================================
# EFFICIENCY RANKING
# ============================================================

print()
print("=" * 100)
print("RANKING BY THROUGHPUT")
print("=" * 100)

efficiency_ranking = results_df.sort_values(
    "throughput_samples_sec",
    ascending=False
)

display(
    efficiency_ranking[
        [
            "model",
            "accuracy",
            "macro_f1",
            "parameters_m",
            "gflops",
            "peak_gpu_memory_mb",
            "latency_per_sample_ms",
            "throughput_samples_sec",
        ]
    ]
)


# ============================================================
# COMPARISON AGAINST HYBRID
# ============================================================

hybrid = results_df[
    results_df["model"] == "Hybrid Spatial-Spectral"
].iloc[0]

comparison_df = results_df.copy()

comparison_df["accuracy_vs_hybrid_pp"] = (
    comparison_df["accuracy"] - hybrid["accuracy"]
) * 100

comparison_df["macro_f1_vs_hybrid_pp"] = (
    comparison_df["macro_f1"] - hybrid["macro_f1"]
) * 100

comparison_df["parameters_reduction_vs_hybrid_pct"] = (
    1 -
    comparison_df["parameters"] /
    hybrid["parameters"]
) * 100

comparison_df["memory_reduction_vs_hybrid_pct"] = (
    1 -
    comparison_df["peak_gpu_memory_mb"] /
    hybrid["peak_gpu_memory_mb"]
) * 100

comparison_df["latency_reduction_vs_hybrid_pct"] = (
    1 -
    comparison_df["latency_per_sample_ms"] /
    hybrid["latency_per_sample_ms"]
) * 100

comparison_df["throughput_gain_vs_hybrid_pct"] = (
    comparison_df["throughput_samples_sec"] /
    hybrid["throughput_samples_sec"] -
    1
) * 100


print()
print("=" * 100)
print("COMPARISON AGAINST HYBRID SPATIAL-SPECTRAL")
print("=" * 100)

display(
    comparison_df[
        [
            "model",
            "accuracy_vs_hybrid_pp",
            "macro_f1_vs_hybrid_pp",
            "parameters_reduction_vs_hybrid_pct",
            "memory_reduction_vs_hybrid_pct",
            "latency_reduction_vs_hybrid_pct",
            "throughput_gain_vs_hybrid_pct",
        ]
    ].sort_values(
        "macro_f1_vs_hybrid_pp",
        ascending=False
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

save_dir = Path("results")
save_dir.mkdir(exist_ok=True)

csv_path = save_dir / "salinas_sota_benchmark_results.csv"
xlsx_path = save_dir / "salinas_sota_benchmark_results.xlsx"

results_df.to_csv(
    csv_path,
    index=False
)

with pd.ExcelWriter(
    xlsx_path,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="Benchmark Results",
        index=False
    )

    macro_f1_ranking.to_excel(
        writer,
        sheet_name="Macro-F1 Ranking",
        index=False
    )

    accuracy_ranking.to_excel(
        writer,
        sheet_name="Accuracy Ranking",
        index=False
    )

    efficiency_ranking.to_excel(
        writer,
        sheet_name="Efficiency Ranking",
        index=False
    )

    comparison_df.to_excel(
        writer,
        sheet_name="Comparisons",
        index=False
    )


print()
print("=" * 100)
print("SAVED")
print("=" * 100)
print("CSV :", csv_path)
print("Excel:", xlsx_path)

SALINAS SOTA BENCHMARK RESULTS


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,Hybrid Spatial-Spectral,0.981000,0.963900,605712,0.605700,2.330000,0.082100,252.450000,128,43.81500,44.8740,45.744000,2.002000,0.342300,2921.360000
1,SpectralFormer (Official),0.927400,0.918700,399381,0.399400,1.553000,0.043300,370.260000,128,67.64100,65.2510,71.436000,18.100000,0.528400,1892.350000
2,SSFTT (Official),0.981500,0.954100,153224,0.153200,0.598000,0.016900,47.740000,128,8.62300,5.6020,10.491000,13.052000,0.067400,14844.050000
3,MorphFormer,0.970756,0.961936,280752,0.280752,1.072556,8.522273,518.783691,128,137.57408,133.4902,159.426660,16.447802,1.074798,930.407821
4,GSC-ViT,0.980627,0.966688,179024,0.179024,0.689804,1.694452,74.940918,64,14.93746,14.7316,16.599685,0.825406,0.233398,4284.530302



RANKING BY MACRO-F1


,model,accuracy,macro_f1,parameters_m,gflops,latency_per_sample_ms,throughput_samples_sec
4,GSC-ViT,0.980627,0.966688,0.179024,1.694452,0.233398,4284.530302
0,Hybrid Spatial-Spectral,0.981000,0.963900,0.605700,0.082100,0.342300,2921.360000
3,MorphFormer,0.970756,0.961936,0.280752,8.522273,1.074798,930.407821
2,SSFTT (Official),0.981500,0.954100,0.153200,0.016900,0.067400,14844.050000
1,SpectralFormer (Official),0.927400,0.918700,0.399400,0.043300,0.528400,1892.350000



RANKING BY ACCURACY


...


RANKING BY THROUGHPUT


,model,accuracy,macro_f1,parameters_m,gflops,peak_gpu_memory_mb,latency_per_sample_ms,throughput_samples_sec
2,SSFTT (Official),0.981500,0.954100,0.153200,0.016900,47.740000,0.067400,14844.050000
4,GSC-ViT,0.980627,0.966688,0.179024,1.694452,74.940918,0.233398,4284.530302
0,Hybrid Spatial-Spectral,0.981000,0.963900,0.605700,0.082100,252.450000,0.342300,2921.360000
1,SpectralFormer (Official),0.927400,0.918700,0.399400,0.043300,370.260000,0.528400,1892.350000
3,MorphFormer,0.970756,0.961936,0.280752,8.522273,518.783691,1.074798,930.407821



COMPARISON AGAINST HYBRID SPATIAL-SPECTRAL


,model,accuracy_vs_hybrid_pp,macro_f1_vs_hybrid_pp,parameters_reduction_vs_hybrid_pct,memory_reduction_vs_hybrid_pct,latency_reduction_vs_hybrid_pct,throughput_gain_vs_hybrid_pct
4,GSC-ViT,-0.037269,0.278798,70.444039,70.314550,31.814837,46.662181
0,Hybrid Spatial-Spectral,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,MorphFormer,-1.024354,-0.196351,53.649259,-105.499581,-213.992843,-68.151552
2,SSFTT (Official),0.050000,-0.980000,74.703489,81.089325,80.309670,408.121218
1,SpectralFormer (Official),-5.360000,-4.520000,34.064209,-46.666667,-54.367514,-35.223663



SAVED
CSV : results\salinas_sota_benchmark_results.csv
Excel: results\salinas_sota_benchmark_results.xlsx


In [50]:
%pip install openpyxl

  Obtaining dependency information for openpyxl from https://files.pythonhosted.org/packages/c0/da/977ded879c29cbd04de313843e76868e6e13408a94ed6b987245dc7c8506/openpyxl-3.1.5-py2.py3-none-any.whl.metadata
  Obtaining dependency information for et-xmlfile from https://files.pythonhosted.org/packages/c1/8b/5fe2cc11fee489817272089c4203e679c63b570a5aaeb18d852ae3cbba6a/et_xmlfile-2.0.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/250.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/250.9 kB ? eta -:--:--
   ----------- --------------------------- 71.7/250.9 kB 653.6 kB/s eta 0:00:01
   ----------- --------------------------- 71.7/250.9 kB 653.6 kB/s eta 0:00:01
   -------------------- ----------------- 133.1/250.9 kB 657.1 kB/s eta 0:00:01
   -------------------- ----------------- 133.1/250.9 kB 657.1 kB/s eta 0:00:01
   -------------------- ----------------- 133.1/250.9 k


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
